In [1]:
import sys, os, json, torch

BACKEND_PATH = os.path.join(os.getcwd(), '..', 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

from app.models.nanogpt import NanoGPT

DATA_DIR    = os.path.join(os.getcwd(), '..', 'data', 'ar')
WEIGHTS_DIR = os.path.join(BACKEND_PATH, 'app', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print('Setup AR v2 OK ✓')

Setup AR v2 OK ✓


In [2]:
# ── Même structure que Sprint3_v2 mais pour l'arabe ─────────

CATEGORIES = [
    ('marque',  'luxe'),
    ('marque',  'tech'),
    ('marque',  'food'),
    ('marque',  'general'),
    ('societe', 'tech'),
    ('societe', 'services'),
    ('societe', 'industrie'),
    ('societe', 'general'),
]

def load_category_ar(type_, secteur):
    path = os.path.join(DATA_DIR, f'{type_}_{secteur}.txt')
    if not os.path.exists(path):
        print(f'  ⚠️  Manquant : {path}')
        return []
    with open(path, encoding='utf-8') as f:
        names = [l.strip() for l in f if l.strip() and ' ' not in l.strip()]
    token = f'#{type_.upper()}#{secteur.upper()}#'
    return [f'{token}{name}' for name in names]

all_lines = []
for type_, secteur in CATEGORIES:
    lines = load_category_ar(type_, secteur)
    all_lines.extend(lines)
    ex = lines[0] if lines else 'vide'
    print(f'  {type_:8s} / {secteur:12s} → {len(lines):4d} noms  (ex: {ex})')

text = '\n'.join(all_lines) + '\n'
print(f'\nTotal AR : {len(all_lines)} noms  |  {len(text)} caractères')

  marque   / luxe         →  146 noms  (ex: #MARQUE#LUXE#أبهة)
  marque   / tech         →  107 noms  (ex: #MARQUE#TECH#ابتكار)
  marque   / food         →  105 noms  (ex: #MARQUE#FOOD#أرضية)
  marque   / general      →  105 noms  (ex: #MARQUE#GENERAL#أمانة)
  societe  / tech         →   89 noms  (ex: #SOCIETE#TECH#ابتكارات)
  societe  / services     →   93 noms  (ex: #SOCIETE#SERVICES#اتزان)
  societe  / industrie    →   62 noms  (ex: #SOCIETE#INDUSTRIE#ابنية)
  societe  / general      →   68 noms  (ex: #SOCIETE#GENERAL#أمانة)

Total AR : 775 noms  |  16324 caractères


In [4]:
# ── Vocabulaire arabe + tokens ASCII ────────────────────────
# IMPORTANT : le vocab arabe contient maintenant '#', lettres
# ASCII majuscules pour les tokens de contrôle ET les lettres
# arabes pour les noms générés.

# ── Vocabulaire arabe + tokens ASCII ────────────────────────

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

assert '#' in stoi, '# absent du vocab !'

# Bornes Unicode arabe
arabic_start = '\u0600'
arabic_end   = '\u06ff'

arabic_count = sum(1 for c in chars if arabic_start <= c <= arabic_end)
ascii_count  = sum(1 for c in chars if c.isascii())

print(f'Vocabulaire AR v2 : {vocab_size} caractères')
print(f'Dont lettres arabes : {arabic_count}')
print(f'Dont ASCII (tokens) : {ascii_count}')

vocab_pkg = {'stoi': stoi}

with open(os.path.join(WEIGHTS_DIR, 'vocab_ar.json'), 'w', encoding='utf-8') as f:
    json.dump(vocab_pkg, f, ensure_ascii=False, indent=2)

print('\nvocab_ar.json sauvegardé ✓')

Vocabulaire AR v2 : 57 caractères
Dont lettres arabes : 36
Dont ASCII (tokens) : 21

vocab_ar.json sauvegardé ✓


In [5]:
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'train: {len(train_data)} tokens | val: {len(val_data)} tokens')

train: 14691 tokens | val: 1633 tokens


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BLOCK_SIZE = 32  # identique FR v2

model_ar = NanoGPT(
    vocab_size = vocab_size,
    n_embed    = 64,
    n_head     = 4,
    n_layer    = 4,
    block_size = BLOCK_SIZE,
    dropout    = 0.1
)
model_ar.to(device)
print(f'NanoGPT AR | params: {model_ar.count_params():,} | block_size: {BLOCK_SIZE}')

NanoGPT AR | params: 208,384 | block_size: 32


In [7]:
import torch.optim as optim

MAX_ITERS     = 4000
EVAL_INTERVAL = 400
BATCH_SIZE    = 64
LR            = 1e-3

optimizer = optim.AdamW(model_ar.parameters(), lr=LR)

def get_batch_ar(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([d[i:i+BLOCK_SIZE]     for i in ix])
    y  = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

print('🚀 Entraînement AR v2 (tokens de contrôle)...')
model_ar.train()

for it in range(MAX_ITERS):
    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
        model_ar.eval()
        with torch.no_grad():
            _, lt = model_ar(*get_batch_ar('train'))
            _, lv = model_ar(*get_batch_ar('val'))
        print(f'Iter {it:5d} | Train: {lt.item():.4f} | Val: {lv.item():.4f}')
        model_ar.train()
    xb, yb = get_batch_ar('train')
    _, loss = model_ar(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('\n✓ Entraînement AR terminé')

🚀 Entraînement AR v2 (tokens de contrôle)...
Iter     0 | Train: 4.0498 | Val: 4.0350
Iter   400 | Train: 0.6809 | Val: 1.0922
Iter   800 | Train: 0.5652 | Val: 0.9827
Iter  1200 | Train: 0.4252 | Val: 0.8769
Iter  1600 | Train: 0.3746 | Val: 0.7847
Iter  2000 | Train: 0.3207 | Val: 0.7612
Iter  2400 | Train: 0.2888 | Val: 0.7988
Iter  2800 | Train: 0.2656 | Val: 0.8597
Iter  3200 | Train: 0.2651 | Val: 0.8822
Iter  3600 | Train: 0.2561 | Val: 0.8543
Iter  3999 | Train: 0.2364 | Val: 0.7797

✓ Entraînement AR terminé


In [8]:
import torch.nn.functional as F

def generate_ar(model, stoi, itos, ctrl_token, n=8, temp=0.9, top_k=15):
    model.eval()
    names, pad = [], stoi.get('\n', 0)
    for _ in range(n * 4):
        ctx_ids = [stoi.get(c, 0) for c in ctrl_token if c in stoi]
        ctx = torch.zeros(1, model.block_size, dtype=torch.long)
        for j, cid in enumerate(ctx_ids[-model.block_size:]):
            ctx[0, -(len(ctx_ids))+j] = cid
        ctx = ctx.to(device)
        chars = []
        with torch.no_grad():
            for _ in range(15):
                logits, _ = model(ctx)
                logits = logits[:, -1, :] / max(temp, 1e-5)
                v, _   = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
                ix  = torch.multinomial(F.softmax(logits, -1), 1).item()
                ch  = itos.get(ix, '')
                if ch in ('\n', '#') or not ch:
                    break
                chars.append(ch)
                new = torch.roll(ctx, -1, dims=1).clone()
                new[0,-1] = ix
                ctx = new
        nm = ''.join(chars).split('#')[-1].strip()
        if 2 <= len(nm) <= 10 and nm not in names:
            names.append(nm)
        if len(names) >= n:
            break
    return names[:n]

tests_ar = [
    '#MARQUE#LUXE#',
    '#MARQUE#TECH#',
    '#MARQUE#FOOD#',
    '#SOCIETE#SERVICES#',
]
for token in tests_ar:
    noms = generate_ar(model_ar, stoi, itos, token, n=5)
    print(f'{token:25s} → {noms}')

#MARQUE#LUXE#             → ['مبهجة', 'زخم', 'فريدة', 'نفيسة', 'زهير']
#MARQUE#TECH#             → ['عبقرية', 'النترنت', 'ربطية', 'مستقبل', 'استشراف']
#MARQUE#FOOD#             → ['فطور', 'مائدة', 'عسل', 'مراعي', 'مائز']
#SOCIETE#SERVICES#        → ['الجودة', 'التجاري', 'تناسب', 'التكوين', 'الشبكة']


In [9]:
model_path = os.path.join(WEIGHTS_DIR, 'model_ar.pt')
torch.save(model_ar.to('cpu').state_dict(), model_path)
print(f'Poids sauvegardés → {model_path} ✓')
for fname in ['model_ar.pt', 'vocab_ar.json']:
    fp   = os.path.join(WEIGHTS_DIR, fname)
    size = os.path.getsize(fp) // 1024
    print(f'  {fname:20s}  {size} Ko')

Poids sauvegardés → c:\Users\Lenovo\Documents\ProjetVersion1\notebooks\..\backend\app\weights\model_ar.pt ✓
  model_ar.pt           847 Ko
  vocab_ar.json         0 Ko
